In [1]:
pip install selenium webdriver-manager pandas beautifulsoup4

Note: you may need to restart the kernel to use updated packages.


In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

"""
T-Forecast V5.7.2 - 6大媒體熱門新聞全自動爬蟲 (全網域結構適配版)
基底: V5.7.1
目標: 解決「子網域 (如 sports.ltn, star.setn) 結構不同導致抓不到」的問題。
方法: 使用「多重選擇器 (Selector List)」機制，自動適配不同版型。
"""

import os
import sys
import time
import random
import logging
import pandas as pd
from datetime import datetime, timedelta, time as dt_time
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import re

# ===========================
# 1. 環境與變數設定
# ===========================

try:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__))
except NameError:
    BASE_DIR = os.getcwd()

DATA_FILENAME = os.path.join(BASE_DIR, 'news_dataset_v5.csv')
LOG_FILENAME = os.path.join(BASE_DIR, 'scraper_v5.log')

MEDIA_TARGETS = {
    "TVBS": "https://news.tvbs.com.tw/hot",
    "UDN": "https://udn.com/rank/pv/2",
    "ChinaTimes": "https://www.chinatimes.com/hotnews/?chdtv",
    "LTN": "https://news.ltn.com.tw/list/breakingnews/popular",
    "SETN": "https://www.setn.com/viewall.aspx?pagegroupid=0",
    "ETtoday": "https://www.ettoday.net/news/hot-news.htm"
}

COLUMNS = ['url', 'media', 'category', 'publish_time', 'title', 'content', 'fetch_time']

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.FileHandler(LOG_FILENAME, encoding='utf-8'), logging.StreamHandler(sys.stdout)]
)

# ===========================
# 2. 輔助工具函式
# ===========================

def format_output_time(dt_obj=None):
    if dt_obj is None: dt_obj = datetime.now()
    return dt_obj.strftime("%Y/%m/%d %I:%M:%S %p").lower()

def clean_text(text):
    if not text: return "N/A"
    return re.sub(r'\s+', ' ', text).strip()

def ensure_newline_in_csv(filename):
    if not os.path.exists(filename): return
    try:
        with open(filename, 'rb+') as f:
            f.seek(0, 2)
            if f.tell() > 0:
                f.seek(-1, 2)
                if f.read(1) != b'\n':
                    logging.info("🔧 自動補上 CSV 換行符號...")
                    f.write(b'\n')
    except Exception as e:
        logging.warning(f"⚠️ CSV 換行檢查失敗: {e}")

def parse_time_string(time_str):
    if not time_str or time_str == "N/A": return "N/A"
    clean = time_str.strip()
    now = datetime.now()

    # ISO 格式
    if 'T' in clean:
        try:
            dt_part = clean.split('T')[0] + ' ' + clean.split('T')[1].split('+')[0].split('.')[0].replace('Z', '')
            dt = datetime.strptime(dt_part, "%Y-%m-%d %H:%M:%S")
            return format_output_time(dt)
        except: pass

    # 一般格式
    clean_nums = re.sub(r'[^\d\-\:\/\s]', '', clean).strip()
    formats = ["%Y/%m/%d %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y/%m/%d %H:%M", "%Y-%m-%d %H:%M"]
    for fmt in formats:
        try:
            dt = datetime.strptime(clean_nums[:19], fmt)
            return format_output_time(dt)
        except: continue

    # 無年份格式
    short_formats = ["%m/%d %H:%M", "%m-%d %H:%M"]
    for fmt in short_formats:
        try:
            dt = datetime.strptime(clean_nums, fmt)
            dt = dt.replace(year=now.year)
            if dt > now + timedelta(days=1): dt = dt.replace(year=now.year - 1)
            return format_output_time(dt)
        except: continue

    # 相對時間
    try:
        if "分鐘前" in clean:
            minutes = int(re.search(r'\d+', clean).group())
            dt = now - timedelta(minutes=minutes)
            return format_output_time(dt)
        elif "小時前" in clean:
            hours = int(re.search(r'\d+', clean).group())
            dt = now - timedelta(hours=hours)
            return format_output_time(dt)
    except: pass
    return "N/A"

def is_sleeping_time():
    now = datetime.now().time()
    return dt_time(1, 0) <= now <= dt_time(5, 0)

def get_driver():
    opts = Options()
    opts.add_argument("--headless") 
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-blink-features=AutomationControlled")
    opts.add_argument("user-agent=Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/125.0.0.0 Safari/537.36")
    opts.add_experimental_option("prefs", {"profile.managed_default_content_settings.images": 2})
    return webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=opts)

def human_scroll(driver):
    try:
        driver.execute_script("window.scrollBy(0, 600);")
        time.sleep(1)
        driver.execute_script("window.scrollBy(0, -200);")
    except: pass

# ===========================
# 3. 核心解析邏輯 (V5.7.2 重點)
# ===========================

def extract_meta_content_regex(html_str, prop_name):
    patterns = [
        fr'<meta[^>]*property=["\']{prop_name}["\'][^>]*content=["\']([^"\']+)["\']',
        fr'<meta[^>]*content=["\']([^"\']+)["\'][^>]*property=["\']{prop_name}["\']',
        fr'<meta[^>]*name=["\']{prop_name}["\'][^>]*content=["\']([^"\']+)["\']',
        fr'<meta[^>]*content=["\']([^"\']+)["\'][^>]*name=["\']{prop_name}["\']',
        fr'<meta[^>]*itemprop=["\']{prop_name}["\'][^>]*content=["\']([^"\']+)["\']'
    ]
    for p in patterns:
        match = re.search(p, html_str, re.IGNORECASE)
        if match: return match.group(1).strip()
    return None

def find_iso_date_in_html(html_str):
    match = re.search(r'202\d-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}', html_str)
    if match: return match.group(0)
    return None

def parse_common_meta_v5_8(soup, html_str):
    title = extract_meta_content_regex(html_str, 'og:title')
    if not title and soup.h1: title = soup.h1.get_text(strip=True)
    
    category = extract_meta_content_regex(html_str, 'article:section') or \
               extract_meta_content_regex(html_str, 'section')
    
    pub_time = extract_meta_content_regex(html_str, 'article:published_time') or \
               extract_meta_content_regex(html_str, 'pubdate') or \
               extract_meta_content_regex(html_str, 'datePublished')
    
    if not pub_time:
        pub_time = find_iso_date_in_html(html_str)

    return title, category, pub_time

def try_selectors(soup, selectors):
    """[核心] 依序嘗試多個選擇器，回傳第一個成功的"""
    for sel in selectors:
        tag = soup.select_one(sel)
        if tag: return clean_text(tag.get_text())
    return None

# --- 媒體專屬解析器 (全子網域適配) ---

def parse_tvbs(soup, html_str):
    title, category, pub_time = parse_common_meta_v5_8(soup, html_str)
    
    if not title: title = try_selectors(soup, ['h1.margin_b20', '.news_detail_div h1', 'h1'])
    if not category: category = try_selectors(soup, ['.breadcrumb li:nth-child(2)', '.news_type'])
    if not pub_time: pub_time = try_selectors(soup, ['.author .time', '.news_time', '#news_time'])
    
    content = try_selectors(soup, ['#news_detail_div', '.article_content', '.news_content']) or "N/A"
    return title or "N/A", content, category or "N/A", parse_time_string(pub_time)

def parse_udn(soup, html_str):
    title, category, pub_time = parse_common_meta_v5_8(soup, html_str)
    
    if not title: title = try_selectors(soup, ['h1.article-content__title', 'h1'])
    if not pub_time: pub_time = try_selectors(soup, ['.article-content__time', '.story_bady_info_author'])
    
    content = try_selectors(soup, ['section.article-content__editor', '.article-content', '#story_body_content']) or "N/A"
    return title or "N/A", content, category or "N/A", parse_time_string(pub_time)

def parse_chinatimes(soup, html_str):
    title, category, pub_time = parse_common_meta_v5_8(soup, html_str)
    
    if not pub_time:
        t = soup.select_one('time[datetime]')
        if t: pub_time = t.get('datetime')
        else: pub_time = try_selectors(soup, ['.meta-info time', '.date'])

    content = try_selectors(soup, ['.article-body', 'div[itemprop="articleBody"]']) or "N/A"
    return title or "N/A", content, category or "N/A", parse_time_string(pub_time)

def parse_ltn(soup, html_str, url):
    # 1. Meta 優先
    title, category, pub_time = parse_common_meta_v5_8(soup, html_str)
    
    if not title: 
        title = try_selectors(soup, ['div.whitecon h1', '.news_content h1', 'h1', '.article-title'])
    
    # [核心修正] LTN 多重結構適配
    # div.text = 主站
    # .news_content = 體育 (sports.ltn) 專用，不帶 .text
    # .whitecon .text = 財經 (ec.ltn)
    content = try_selectors(soup, [
        'div.text', 
        '.news_content', 
        '.whitecon .text',
        'div[itemprop="articleBody"]',
        '.news_p'
    ]) or "N/A"
    
    if content != "N/A":
        content = content.replace("請繼續往下閱讀...", "").strip()

    if not pub_time: 
        pub_time = try_selectors(soup, ['span.time', '.c_time', '.mobile_time', '.news_time'])

    return title or "N/A", content, category or "N/A", parse_time_string(pub_time)

def parse_setn(soup, html_str):
    # 1. Meta 優先
    title, category, pub_time = parse_common_meta_v5_8(soup, html_str)
    
    if not title: 
        title = try_selectors(soup, ['h1.news-title-3', 'h1.page-title-text', 'h1.title', 'h1'])
        
    # [核心修正] SETN 多重結構適配
    # article = 主站
    # .page-text = 娛樂星聞 (star.setn)
    # .news-content = 健康網 (health.setn)
    content = try_selectors(soup, [
        'article', 
        'div#Content1', 
        '.page-text', 
        '.news-content',
        '.Content2'
    ]) or "N/A"
    
    if not pub_time: 
        pub_time = try_selectors(soup, ['time.page-date', '.news-time', '.date', '.time'])
        
    return title or "N/A", content, category or "N/A", parse_time_string(pub_time)

def parse_ettoday(soup, html_str):
    title, category, pub_time = parse_common_meta_v5_8(soup, html_str)
    
    # ETtoday 多重結構適配
    content = try_selectors(soup, [
        'div.story', 
        '.fncnews-content', 
        'article'
    ]) or "N/A"
    
    if content != "N/A": content = content.replace("請繼續往下閱讀...", "")

    if not pub_time:
        t = soup.select_one('time.date')
        if t: pub_time = t.get('datetime') or t.get_text()
    
    return title or "N/A", content, category or "N/A", parse_time_string(pub_time)

# ===========================
# 4. 主解析路由器 & 連結獲取
# ===========================

def parse_article_router(driver, media_name, url):
    html_str = driver.page_source
    soup = BeautifulSoup(html_str, 'html.parser')
    try:
        if media_name == "TVBS": return parse_tvbs(soup, html_str)
        elif media_name == "UDN": return parse_udn(soup, html_str)
        elif media_name == "ChinaTimes": return parse_chinatimes(soup, html_str)
        elif media_name == "LTN": return parse_ltn(soup, html_str, url)
        elif media_name == "SETN": return parse_setn(soup, html_str)
        elif media_name == "ETtoday": return parse_ettoday(soup, html_str)
    except Exception as e:
        logging.error(f"解析錯誤 {url}: {e}")
    return "N/A", "N/A", "N/A", "N/A"

def get_hot_links(driver, media_name):
    soup = BeautifulSoup(driver.page_source, 'html.parser')
    links = set()
    try:
        items = []
        if media_name == "TVBS": items = soup.select('.news_list .list li > a')
        elif media_name == "UDN": items = soup.select('.story-list__news h2 a')
        elif media_name == "ChinaTimes": items = soup.select('.hot-news .vertical-list li h3 a')
        elif media_name == "LTN": 
            items = soup.select('ul.list li a.tit')
            if not items: items = soup.select('ul.list li > a')
        elif media_name == "SETN": items = soup.select('.NewsList h3 a')
        elif media_name == "ETtoday": items = soup.select('.part_pictxt_3 h3 a')

        for item in items:
            href = item.get('href')
            if href:
                if href.startswith('//'): href = 'https:' + href
                elif href.startswith('/'): 
                    domain = MEDIA_TARGETS[media_name].split('/')[2]
                    href = f"https://{domain}" + href
                
                # 過濾非內文連結
                if re.search(r'\d+', href) and not any(x in href for x in ['search', 'category', 'photo']):
                    links.add(href)
    except Exception as e:
        logging.error(f"[{media_name}] 列表解析錯誤: {e}")
    
    return sorted(list(links), reverse=True)[:10] # 維持 V5.7 邏輯 (前10筆)

# ===========================
# 5. 主執行程序
# ===========================

def run_spider():
    if is_sleeping_time():
        logging.info("💤 半夜休眠中...")
        return

    logging.info(f"======= V5.7.2 (結構適配版) 開始: {format_output_time()} =======")
    driver = get_driver()
    all_data = []

    try:
        for media_name, url in MEDIA_TARGETS.items():
            logging.info(f"🚀 [{media_name}] 掃描熱門榜: {url}")
            try:
                driver.get(url)
                time.sleep(3)
                links = get_hot_links(driver, media_name)
                logging.info(f"   -> 取得 {len(links)} 篇連結")

                for i, link in enumerate(links):
                    logging.info(f"   ({i+1}/{len(links)}) 分析: {link}")
                    try:
                        driver.get(link)
                        human_scroll(driver)
                        time.sleep(random.uniform(3, 5))
                        
                        title, content, category, pub_time = parse_article_router(driver, media_name, link)
                        
                        if content != "N/A" and len(content) > 50:
                            if title == "N/A" or not title: 
                                title = driver.title.split('|')[0].strip()

                            all_data.append({
                                'url': link,
                                'media': media_name,
                                'category': category,
                                'publish_time': pub_time,
                                'title': title,
                                'content': content,
                                'fetch_time': format_output_time()
                            })
                        else:
                            logging.warning(f"   ⚠️ 抓取失敗 (Len: {len(content)})")
                    except Exception as e:
                        logging.error(f"   🔥 單篇錯誤: {e}")
            except Exception as e:
                logging.error(f"   🔥 列表錯誤: {e}")
    finally:
        driver.quit()

    if all_data:
        ensure_newline_in_csv(DATA_FILENAME)
        
        df = pd.DataFrame(all_data, columns=COLUMNS)
        is_exist = os.path.exists(DATA_FILENAME)
        df.to_csv(DATA_FILENAME, mode='a', header=not is_exist, index=False, encoding='utf-8-sig')
        logging.info(f"✅ 完成! 寫入 {len(all_data)} 筆資料到 {DATA_FILENAME}")
    else:
        logging.info("⚠️ 本次無有效資料寫入")

if __name__ == "__main__":
    run_spider()

2025-12-01 13:04:14,126 - INFO - ======= V5.7.2 (結構適配版) 開始: 2025/12/01 01:04:14 pm =======
2025-12-01 13:04:14,127 - INFO - ====== WebDriver manager ======
2025-12-01 13:04:14,401 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-01 13:04:14,742 - INFO - Get LATEST chromedriver version for google-chrome
2025-12-01 13:04:14,952 - INFO - Driver [/Users/xiaobing/.wdm/drivers/chromedriver/mac64/142.0.7444.175/chromedriver-mac-arm64/chromedriver] found in cache
2025-12-01 13:04:15,804 - INFO - 🚀 [TVBS] 掃描熱門榜: https://news.tvbs.com.tw/hot
2025-12-01 13:04:19,533 - INFO -    -> 取得 10 篇連結
2025-12-01 13:04:19,534 - INFO -    (1/10) 分析: https://news.tvbs.com.tw/world/3059686
2025-12-01 13:04:26,187 - INFO -    (2/10) 分析: https://news.tvbs.com.tw/world/3059646
2025-12-01 13:04:31,604 - INFO -    (3/10) 分析: https://news.tvbs.com.tw/world/3059645
2025-12-01 13:04:36,707 - INFO -    (4/10) 分析: https://news.tvbs.com.tw/world/3059615
2025-12-01 13:04:42,730 - INFO -    (5/10) 分析: http